In [ ]:
# US Airline Routes & Fares (1993–2024) — Full EDA Notebook
# Setup and load dataset
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8")
sns.set(font_scale=1.1)

# Load Dataset
file_path = "US Airline Flight Routes and Fares 1993-2024.csv"

df = pd.read_csv(file_path, low_memory=False)
print("Shape:", df.shape)
df.head()

# Inspect Dataset
print("Columns:", df.columns.tolist())
df.info()
df.describe()

# Standardize Columns
df.columns = (
    df.columns
      .str.strip()
      .str.lower()
      .str.replace(" ", "_")
)

df.head()

# Handle Missing Values (Median Impute Numeric)
numeric_cols = df.select_dtypes(include=["int64","float64"]).columns
non_numeric_cols = df.select_dtypes(exclude=["int64","float64"]).columns

# Median impute numeric
for col in numeric_cols:
    df[col] = df[col].fillna(df[col].median())

# Fill non-numeric with "Unknown"
for col in non_numeric_cols:
    df[col] = df[col].fillna("Unknown")

df.isna().sum()

# Executive Visualizations
# Fare Trend Over Time (1993–2024)
fare_year = df.groupby("year")["fare"].mean().reset_index()

plt.figure(figsize=(12,6))
ax = sns.lineplot(data=fare_year, x="year", y="fare", marker="o", color="tab:blue")

# Add labels on each point
for x, y in zip(fare_year["year"], fare_year["fare"]):
    ax.text(x, y, f"{y:.0f}", ha="center", va="bottom", fontsize=8, color="tab:blue")

plt.title("Average Fare Trend (1993–2024)")
plt.xlabel("Year")
plt.ylabel("Average Fare")
plt.grid(True, linestyle="--", alpha=0.4)

# --- Bottom-right summary box ---
summary_text = (
    "Summary:\n"
    "Average fares show a long-term trend shaped by market\n"
    "competition and economic cycles. The pattern highlights\n"
    "periods of fare increases and decreases across 1993–2024."
)

# Add annotation box
ax.annotate(
    summary_text,
    xy=(0.98, 0.02),          # bottom-right corner (axes coordinates)
    xycoords="axes fraction",
    ha="right",
    va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.8)
)

plt.tight_layout()
plt.show()


# Passenger Trend Over Time
pass_year = df.groupby("year")["passengers"].sum().reset_index()

plt.figure(figsize=(12,6))
ax = sns.lineplot(data=pass_year, x="year", y="passengers", marker="o", color="tab:green")

# Add labels on each point
for x, y in zip(pass_year["year"], pass_year["passengers"]):
    ax.text(x, y, f"{y:,}", ha="center", va="bottom", fontsize=8, color="tab:green")

plt.title("Passenger Trend (1993–2024)")
plt.xlabel("Year")
plt.ylabel("Total Passengers")
plt.grid(True, linestyle="--", alpha=0.4)

# --- Bottom-center summary box ---
summary_text = (
    "Summary:\n"
    "Passenger volume shows a long-term upward trend,\n"
    "reflecting growing demand for air travel and increased\n"
    "network utilization across the U.S. airline industry."
)

ax.annotate(
    summary_text,
    xy=(0.5, 0.02),            # bottom-center (axes fraction)
    xycoords="axes fraction",
    ha="center",
    va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.8)
)

plt.tight_layout()
plt.show()


# Market Share Trend (Large vs Low‑Fare Carriers)
ms_year = df.groupby("year")[["large_ms","lf_ms"]].mean().reset_index()

plt.figure(figsize=(12,6))
ax = sns.lineplot(
    data=ms_year, x="year", y="large_ms",
    marker="o", color="tab:red", label="Large Carrier MS"
)

sns.lineplot(
    data=ms_year, x="year", y="lf_ms",
    marker="o", color="tab:purple", label="Low-Fare Carrier MS"
)

plt.title("Market Share Trend (1993–2024)")
plt.xlabel("Year")
plt.ylabel("Market Share")
plt.grid(True, linestyle="--", alpha=0.4)

# --- Bottom-right summary box ---
summary_text = (
    "Summary:\n"
    "Large carriers show a gradual decline in market share,\n"
    "while low‑fare carriers steadily expand. This reflects\n"
    "a long-term competitive shift in the U.S. airline market."
)

ax.annotate(
    summary_text,
    xy=(0.98, 0.02),            # bottom-right corner (axes fraction)
    xycoords="axes fraction",
    ha="right",
    va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85)
)

plt.tight_layout()
plt.show()

# Fare Distribution (Histogram + Labels)
import numpy as np

plt.figure(figsize=(10,6))
ax = sns.histplot(df["fare"], bins=40, kde=True, color="tab:orange")

# Add labels on each bar
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(
            p.get_x() + p.get_width()/2,
            height,
            f"{int(height)}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.title("Distribution of Fare")
plt.xlabel("Fare")
plt.ylabel("Count")

# ---- Skewness calculation ----
fare_skew = df["fare"].skew()

# Interpret skewness
if fare_skew > 1:
    skew_text = "high positive skew (long right tail)"
elif fare_skew > 0.5:
    skew_text = "moderate positive skew"
elif fare_skew > 0:
    skew_text = "slight positive skew"
elif fare_skew < -1:
    skew_text = "high negative skew (long left tail)"
elif fare_skew < -0.5:
    skew_text = "moderate negative skew"
elif fare_skew < 0:
    skew_text = "slight negative skew"
else:
    skew_text = "approximately symmetric"

# ---- Bottom-right summary box ----
summary_text = (
    f"Summary:\n"
    f"Fare distribution shows {skew_text}.\n"
    f"Skewness value: {fare_skew:.2f}\n"
    f"Most fares cluster in mid-range values\n"
    f"with fewer high-priced outliers."
)

ax.annotate(
    summary_text,
    xy=(0.98, 0.02),          # bottom-right corner
    xycoords="axes fraction",
    ha="right",
    va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85)
)

plt.tight_layout()
plt.show()

# Distance vs Fare (Scatter + Trend Line)
plt.figure(figsize=(12,7))

# Scatter + regression line
sns.scatterplot(
    data=df.sample(5000),
    x="nsmiles",
    y="fare",
    alpha=0.4,
    color="tab:cyan"
)

sns.regplot(
    data=df,
    x="nsmiles",
    y="fare",
    scatter=False,
    color="tab:red"
)

plt.title("Distance vs Fare")
plt.xlabel("Route Distance (Miles)")
plt.ylabel("Fare")

# ---- Top-right summary box ----
summary_text = (
    "Summary:\n"
    "Longer routes generally have higher fares,\n"
    "as shown by the upward trend. The scatter\n"
    "indicates variability driven by route type,\n"
    "competition, and carrier pricing strategy."
)

plt.annotate(
    summary_text,
    xy=(0.98, 0.98),          # top-right corner
    xycoords="axes fraction",
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85)
)

plt.tight_layout()
plt.show()

# Correlation Heatmap (Key Numeric Features)
key_numeric = ["fare","nsmiles","passengers","large_ms","lf_ms","fare_lg","fare_low"]
corr = df[key_numeric].corr()

# Fare by Carrier Type (Large vs Low‑Fare)
plt.figure(figsize=(10,6))
ax = sns.boxplot(
    data=df,
    x="carrier_lg",
    y="fare",
    hue="carrier_lg",
    palette="tab10",
    legend=False
)

# Label medians
medians = df.groupby("carrier_lg")["fare"].median()
for i, (carrier, med) in enumerate(medians.items()):
    ax.text(i, med, f"{med:.0f}", ha="center", va="bottom", fontsize=8)

plt.title("Fare Distribution by Large Carrier")
plt.xlabel("Large Carrier")
plt.ylabel("Fare")

# ---- Top-right summary box ----
summary_text = (
    "Summary:\n"
    "Fare levels vary significantly across large carriers,\n"
    "reflecting differences in route mix, pricing strategy,\n"
    "and competitive positioning. Wider boxes indicate\n"
    "greater fare variability, while higher medians point\n"
    "to consistently higher-priced networks."
)

plt.annotate(
    summary_text,
    xy=(0.98, 0.98),          # top-right corner
    xycoords="axes fraction",
    ha="right",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85)
)

plt.tight_layout()
plt.show()


plt.figure(figsize=(10,6))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f")
plt.title("Correlation Heatmap")
plt.tight_layout()

# ---- Summary under the chart ----
summary_text = (
    "Summary:\n"
    "Fare is strongly correlated with distance (nsmiles) and carrier-specific fare metrics.\n"
    "Passengers show moderate correlation with distance, reflecting route demand patterns.\n"
    "Large and low‑fare market shares are inversely related, indicating competitive segmentation."
)

plt.annotate(
    summary_text,
    xy=(0.5, -0.25),          # centered below the chart
    xycoords="axes fraction",
    ha="center",
    va="top",
    fontsize=9,
    bbox=dict(boxstyle='round,pad=0.4', fc='white', ec='gray', alpha=0.85)
)

plt.show()

# Executive Summary
Summary = """
This 1993–2024 airline dataset reveals strong structural trends in pricing, demand, and carrier competition. Average fares show a long-term [increase/decrease depending on your plot], influenced by market competition and economic cycles. Passenger volume generally rises, reflecting expanding demand for air travel.

Market share analysis shows a shift toward low‑fare carriers, while traditional large carriers lose share over time. Fares are right‑skewed, with most routes priced in a moderate band and a smaller set of premium long‑distance routes.

Distance is a major driver of fare pricing, with a strong positive correlation. Carrier-level fare differences highlight distinct pricing strategies. Overall, the dataset shows an industry undergoing continuous transformation in pricing, route structure, and competitive dynamics.
"""
print(Summary)


: 

In [ ]:
# Revenue Distribution Histogram + Summary Box
import numpy as np

# ---- Compute Revenue ----
df["revenue"] = df["passengers"] * df["fare"]

plt.figure(figsize=(10,6))
ax = sns.histplot(df["revenue"], bins=40, kde=True, color="tab:green")

# Add labels on each bar
for p in ax.patches:
    height = p.get_height()
    if height > 0:
        ax.text(
            p.get_x() + p.get_width()/2,
            height,
            f"{int(height)}",
            ha="center",
            va="bottom",
            fontsize=8
        )

plt.title("Distribution of Revenue (Passengers × Fare)")
plt.xlabel("Revenue")
plt.ylabel("Count")

# ---- Skewness calculation ----
rev_skew = df["revenue"].skew()

# Interpret skewness
if rev_skew > 1:
    skew_text = "high positive skew (long right tail)"
elif rev_skew > 0.5:
    skew_text = "moderate positive skew"
elif rev_skew > 0:
    skew_text = "slight positive skew"
elif rev_skew < -1:
    skew_text = "high negative skew (long left tail)"
elif rev_skew < -0.5:
    skew_text = "moderate negative skew"
elif rev_skew < 0:
    skew_text = "slight negative skew"
else:
    skew_text = "approximately symmetric"

# ---- Bottom-right summary box ----
summary_text = (
    f"Summary:\n"
    f"Revenue distribution shows {skew_text}.\n"
    f"Skewness value: {rev_skew:.2f}\n"
    f"Revenue is driven by high-passenger,\n"
    f"high-fare routes, creating a long tail."
)

ax.annotate(
    summary_text,
    xy=(0.98, 0.02),          # bottom-right corner
    xycoords="axes fraction",
    ha="right",
    va="bottom",
    fontsize=9,
    bbox=dict(boxstyle="round,pad=0.4", fc="white", ec="gray", alpha=0.85)
)

plt.tight_layout()
plt.show()


: 